In [10]:
import pandas as pd 
import sqlite3
import matplotlib.pyplot as plt # For plotting 
import seaborn as sns 
import plotly.express as px # For plotting 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path 

PROJECT_ROOT = Path.cwd().parent

DATA = 'covid.db'

conn = sqlite3.connect(DATA)

In [11]:
# -------------------------- SQL Query --------------------------
query = """
SELECT 
    country, 
    date,
    cumulative_total_cases AS total_cases,
    cumulative_total_deaths AS total_deaths,
    MIN(date(date)) AS count_begin,
    MAX(date(date)) AS count_end
FROM covid_cases
GROUP BY country  

"""

df = pd.read_sql(query, conn)

# -------------------------- Plotting Total Cases By Country--------------------------

fig = px.bar(df.sort_values('total_cases', ascending=False).dropna(), 
             x='country', y='total_cases',
             hover_data=['total_cases'])
fig.update_layout(
    xaxis=dict(tickangle=90, tickfont=dict(size=7)),
    height=600,
    width=1400,
    title="Total COVID Cases by Country"
)

fig.update_yaxes(type='log')
fig.write_html(PROJECT_ROOT/ "figures" / "Total_COVID_by_Country.html")

fig.show()

In [12]:
# Plot of USA Daily New Cases and Daily Death Rate 

# ------------- SQL query -----------------
query = """
SELECT date, daily_new_cases, daily_new_deaths, death_rate
FROM covid_cases 
WHERE country == 'USA'
"""

df_plot = pd.read_sql(query, conn)

df_plot['date'] = pd.to_datetime(df_plot['date'])

# ------------ Daily New Cases -------------

df_plot['cases_7d'] = df_plot['daily_new_cases'].rolling(7).mean()

peak_idx = df_plot['cases_7d'].idxmax()

peak_date = df_plot.loc[peak_idx, 'date']
peak_case = df_plot.loc[peak_idx, 'cases_7d']

# ------------ Daily Death Rate -------------
df_plot['death_rate7d'] = df_plot['death_rate'].rolling(7).mean()

peak_death_idx = df_plot['death_rate7d'].idxmax()

peak_death = df_plot.loc[peak_death_idx, 'death_rate7d']
peak_death_date = df_plot.loc[peak_death_idx, 'date']
# ------------ Plotting --------------------
fig = make_subplots(rows=1, cols=2, subplot_titles=("Daily Cases", "Death Rate"), column_widths=[0.7, 0.7])

fig.add_trace(go.Scatter( 
              x=df_plot['date'],
              y=df_plot['cases_7d'], 
              mode='lines',
              name='Daily new cases (USA)',
              line=dict(color='#1f77b4')),
              row=1, col=1)

fig.add_trace(go.Scatter( 
              x=df_plot['date'], 
              y=df_plot['death_rate7d'], 
              mode='lines',
              name='Daily Death Rate (USA)', 
              line=dict(color='#e377c2')),
              row=1, col=2)

fig.add_trace(go.Scatter(x=[peak_date], y=[peak_case], 
                         mode='markers', 
                         marker=dict(color='purple', size=8, symbol='star'),
                         name=f"Peak Case: ({peak_date.strftime('%Y-%m-%d')}, {round(peak_case)})"),
             row=1, col=1)

fig.add_trace(go.Scatter(x=[peak_death_date], y=[peak_death], 
                         mode='markers', 
                         marker=dict(color='black', size=8, symbol='star'),
                         name=f"Peak Death rate (%): ({peak_death_date.strftime('%Y-%m-%d')}, {round(peak_death, 3)}%)"),
              row=1, col=2)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_yaxes(type='log', title_text="Cases", row=1, col=1)

fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(type='log', title_text="Death Rate", row=1, col=2)

fig.update_layout(
    width=1500,
    height=500,
    title='Daily New Cases / Death Rate (USA)'
)
fig.write_html(PROJECT_ROOT/ "figures" / "covid_daily_cases_deaths_USA.html")

fig.show()

From this plot we observe that intitially (2020-02 - 2022-2), the daily COVID-cases remained stable however between 2022-01 and 2022-04 the daily new cases spiked with peak 824151 cases. 

Initially, the death rate was quiet high, however there were not much people is diagnosed at this point. The first death reported to be March 3rd, 2020.

## Total death rate for listed countries

In [13]:
query="""
SELECT
    country,
    100.0 * MAX(cumulative_total_deaths) / NULLIF(MAX(cumulative_total_cases), 0)
        AS total_death_rate_pct
FROM covid_cases
GROUP BY country;
"""

df = pd.read_sql(query, conn)


fig = px.bar(df.sort_values('total_death_rate_pct', ascending=False), 
       x='country', 
       y='total_death_rate_pct',
       hover_data=['total_death_rate_pct'],
       color_discrete_sequence=['orange'])
    
    
fig.update_layout(
    xaxis=dict(tickangle=90, tickfont=dict(size=7)),
    height=600,
    width=1400,
    title="Overall Death Rate By Country"
)
fig.update_yaxes(type='log')
fig.write_html(PROJECT_ROOT/ "figures" / "Total_Death_Rate_By_Country.html")

fig.show()

In [14]:
# Pie Chart 

query = """
SELECT country, 
       MAX(cumulative_total_cases) AS total_cases,
       MAX(cumulative_total_deaths) AS total_deaths 
       
FROM covid_cases
GROUP BY country
"""

df = pd.read_sql(query, conn)

# Sorting the df 

df_case_sorted = df.sort_values('total_cases', ascending=False)
df_death_sorted = df.sort_values('total_deaths', ascending=False)


df_case_sorted.loc[df_case_sorted.index[20:], 'country'] = 'Other countries'
df_death_sorted.loc[df_death_sorted.index[20:], 'country'] = 'Other countries'

# ---------------- Plot ----------------
fig = make_subplots(rows=1, cols=2, 
                    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
                    subplot_titles=("Total Cases ", "Total Deaths"))

fig.add_trace(go.Pie(values=df_case_sorted['total_cases'],
                     labels=df_case_sorted['country'],
                     name='Cases',
                     scalegroup='one'
                    ),
                     1, 1)

fig.add_trace(go.Pie(values=df_death_sorted['total_deaths'],
                     labels=df_death_sorted['country'],
                     scalegroup='two',
                     name='Deaths',
                    ),
                     1, 2)

fig.update_traces(textinfo='percent+label')
fig.update_layout(width=1200, 
                  height=800,
                  legend=dict(
                  x=1.1,  # Move right (outside chart)
                  y=0.5),
                  margin=dict(r=150),
                  title='Distributions By Country (Top 20)')



fig.update_yaxes(type='log', row=1, col=1)
fig.update_yaxes(type='log', row=1, col=2)
fig.write_html(PROJECT_ROOT/ "figures" / "Cases_Deaths_Distribution_by_Country_PIECHART.html")

fig.show()

## Choropleth Map

In [15]:
import numpy as np 

query = """
SELECT
    country,
    country_ISO3, 
    MAX(cumulative_total_deaths) AS total_death,
    MAX(cumulative_total_cases) AS total_cases
    
FROM covid_cases
GROUP BY country
 """

df = pd.read_sql(query, conn)

df['log_total_death'] = np.log(df['total_death'])
df['log_total_cases'] = np.log(df['total_cases'])

In [16]:


fig = px.choropleth(
    df,
    locations='country_ISO3',
    color='log_total_death',
    hover_name='country',
    hover_data={'total_death':True, 'log_total_death':False},
    color_continuous_scale='Purples',
    title='Total COVID-19 Deaths by Country'
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=False),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.write_html(PROJECT_ROOT/'Figures/Deaths_Distribution_World_Map.html')
fig.show()

In [17]:
fig = px.choropleth(
    df,
    locations='country_ISO3',
    color='log_total_cases',
    hover_name='country',
    hover_data={'total_cases':True, 'log_total_cases':False},
    color_continuous_scale='Reds',
    title='Total COVID-19 Deaths by Country'
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=False),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.write_html(PROJECT_ROOT/'Figures/Cases_Distribution_World_Map.html')
fig.show()

In [18]:
conn.close()